In [ ]:
# Importing modules
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS

In [ ]:
water_path = PATHS['definitive_notebooks']/ 'water_definitive.parquet'
water_non_merged_pd = pd.read_parquet(water_path)
water_non_merged_pd.head()

In [ ]:
reservoirs_path = PATHS['definitive_notebooks']/ 'reservoirs_merged.parquet'
reservoirs_pd = pd.read_parquet(reservoirs_path)
reservoirs_pd.head()

In [ ]:
reservoirs_pd.isna().sum()

Merging the dataframes

In [ ]:
water_pd = pd.merge(water_non_merged_pd, reservoirs_pd[['id', 'capacity', 'crest_elevation', 'province', 'autonomous_community']], on='id', how='left')
water_pd.head()

Ensuring that we have data for every week:

In [ ]:
water_pd.groupby(['id'])['date'].diff().value_counts()

### Adding year and month features (day is not significant as dates are indexed once a week)

In [ ]:
water_pd['year'] = pd.to_datetime(water_pd['date']).dt.year
water_pd['month'] = pd.to_datetime(water_pd['date']).dt.month
water_pd
water_pd.head()

### Adding lag features

In [ ]:
for lag in [1,2,3,4]:
    water_pd[f'storage_last_week_{lag}'] = water_pd.groupby('id')['storage'].shift(lag)
water_pd['storage_last_year'] = water_pd.groupby('id')['storage'].shift(52)
water_pd.head(15)

In [ ]:
water_pd.isna().sum()

### Rolling average and standard deviation for last month

In [ ]:
water_pd['storage_mean_4w'] = water_pd.groupby('id')['storage'].rolling(4, min_periods=1).mean().reset_index(level=0, drop=True)
water_pd['storage_std_4w'] = water_pd.groupby('id')['storage'].rolling(4, min_periods=1).std().reset_index(level=0, drop=True)

In [ ]:
water_pd.head(15)

In [ ]:
water_engineered_path = PATHS['engineered_data'] / 'water_engineered.parquet'
water_pd_full = pd.read_parquet(water_engineered_path)
water_pd_full.head()

In [ ]:
engineered_water_path = PATHS['engineered_data_notebooks'] / 'water_engineered.parquet'
engineered_water_path.parent.mkdir(parents=True, exist_ok=True)
water_pd.to_parquet(engineered_water_path, index=False)